# Sessió 4 — El model callback
### Teoria, mapa mental i exercicis conceptuals

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

⚠️ **Nota important:** El codi de temps real (`sd.Stream` amb callback) **no funciona a Colab** perquè Colab s'executa en un servidor remot de Google — el so hauria de viatjar per la xarxa, cosa incompatible amb les latències de ~10ms necessàries per a temps real. Per contrast, WebAudio API (navegador) sí que és temps real perquè s'executa localment al teu ordinador.

**El codi real d'avui s'executa a Thonny** (entorn local). Aquest notebook conté la teoria i els exercicis conceptuals per acompanyar la sessió.

## 1. El mapa mental: blocking vs. callback

Executa la cel·la per veure el mapa mental interactiu.

In [ ]:
print("""
╔══════════════════════════════════╦══════════════════════════════════╗
║           BLOCKING               ║        CALLBACK (streaming)      ║
╠══════════════════════════════════╬══════════════════════════════════╣
║  Tu crides sd.rec()              ║  Tu defineixes process()         ║
║  Tu esperes (sd.wait())          ║  El sistema crida process()      ║
║  Tu processes l'array sencer     ║  Processa bloc a bloc (~10ms)    ║
╠══════════════════════════════════╬══════════════════════════════════╣
║  Control: TU                     ║  Control: EL SISTEMA             ║
║  Latència: alta (>1s)            ║  Latència: baixa (~10ms)         ║
║  Resposta a altres inputs: ❌    ║  Resposta a altres inputs: ✅    ║
║  Risc: cap                       ║  Risc: callback lenta → glitch   ║
╠══════════════════════════════════╬══════════════════════════════════╣
║  Quan usar-lo:                   ║  Quan usar-lo:                   ║
║  - Efectes offline               ║  - Efectes en temps real         ║
║  - Processar fitxers             ║  - Sintetitzadors                ║
║  - Exportar WAV                  ║  - Loopers, sequenciadors        ║
╚══════════════════════════════════╩══════════════════════════════════╝
""")

## 2. Exercici conceptual: ordenació (Parsons)

Les línies d'un callback i la seva obertura de Stream estan desordenades. Ordena-les correctament.

```
A.  sd.sleep(10000)
B.  with sd.Stream(samplerate=44100, blocksize=1024, channels=1,
C.  outdata[:] = indata * gain
D.  def process(indata, outdata, frames, time, status):
E.                dtype='float32', callback=process):
```

**Quin és l'ordre correcte?**

In [ ]:
# Escriu l'ordre correcte de les lletres (ex: "A, B, C, D, E")
# i explica per quin motiu cada línia va on va

ordre = ""  # <-- escriu aquí: ex. "D, C, B, E, A"
explicacio = ""  # <-- explica breument

# Solució correcta: D, C, B, E, A
# D: primer definim la funció callback
# C: el cos del callback (indentat dins D)
# B+E: obrim el Stream passant la funció com a callback
# A: mantenim el stream actiu mentre dura

solucio = ["D", "C", "B", "E", "A"]
resposta = [x.strip() for x in ordre.split(",")]

if resposta == solucio:
    print("✅ Correcte!")
elif ordre:
    print(f"❌ Ordre incorrecte. Has posat: {ordre}")
    print("Pista: primer cal definir la funció, després obrir el Stream que la usa.")
else:
    print("Escriu l'ordre a la variable 'ordre' i torna a executar.")

## 3. Exercici conceptual: predicció de comportament

Per a cada callback, prediu qué farà:

In [ ]:
# Callback A
def callback_A(indata, outdata, frames, time, status):
    outdata[:] = indata

# Callback B
def callback_B(indata, outdata, frames, time, status):
    outdata[:] = np.zeros_like(indata)

# Callback C
import numpy as np
gain = 0.0
def callback_C(indata, outdata, frames, time, status):
    outdata[:] = indata * gain

# Callback D
def callback_D(indata, outdata, frames, time, status):
    outdata[:] = indata * 5.0  # amplifica molt

# Preguntes:
print("Callback A → ?")
print("Callback B → ?")
print("Callback C → ? (gain=0.0)  / Qué passaria si canviem gain=0.5 des de fora?")
print("Callback D → ? Quin risc té amplificar tant?")

In [ ]:
# Escriu les teves prediccions aquí (edita i executa)
prediccions = {
    'A': '',  # <-- escriu
    'B': '',  # <-- escriu
    'C': '',  # <-- escriu
    'D': '',  # <-- escriu
}

solucions = {
    'A': 'Pass-through: copia l entrada a la sortida sense canvis',
    'B': 'Silenci: envia zeros (muta el microfon)',
    'C': 'Silenci (gain=0). Si canviem gain=0.5 des de fora, el volum sera al 50%. La variable global es llegida en temps real',
    'D': 'Amplifica x5 → clipping molt probable (valors > 1.0 es distorsionen)',
}

for k in ['A','B','C','D']:
    if prediccions[k]:
        print(f"\nCallback {k}:")
        print(f"  La teva predicció: {prediccions[k]}")
        print(f"  Solució:           {solucions[k]}")
    else:
        print(f"Callback {k}: escriu la teva predicció")

## 4. Exercici: latència i blocksize

Calcula la latència aproximada per a cada combinació de `blocksize` i `sample_rate`.

Fórmula: `latència (ms) = blocksize / sample_rate * 1000`

In [ ]:
import numpy as np

combinacions = [
    (256,  44100),
    (512,  44100),
    (1024, 44100),
    (256,  22050),
    (512,  48000),
]

print(f"{'blocksize':>10} {'sample_rate':>12} {'latència (ms)':>15}")
print("-" * 40)
for blocksize, sr in combinacions:
    latencia_ms = blocksize / sr * 1000
    print(f"{blocksize:>10} {sr:>12} {latencia_ms:>14.1f}")

print("\nPer referència:")
print("  < 10ms: temps real 'transparent' (no es percep)")
print("  10-25ms: acceptable per a la majoria d'aplicacions")
print("  > 40ms: perceptible i molest per a músics")

## 5. Reflexió: el glitch en teoria

Sense executar codi de temps real, raona sobre aquest escenari:

**Situació:** `blocksize=512`, `sample_rate=44100`. El callback aplica un eco complex que tarda ~20ms a calcular.

**Preguntes:**
1. Quant temps té el sistema per processar cada bloc?
2. El callback és prou ràpid?
3. Qué passaria?
4. Com ho solucionaries sense simplificar el càlcul de l'eco?

In [ ]:
# Calcula i raona
blocksize = 512
sample_rate = 44100
temps_disponible_ms = blocksize / sample_rate * 1000
temps_callback_ms = 20  # el nostre eco complex

print(f"Temps disponible per callback: {temps_disponible_ms:.1f} ms")
print(f"Temps que tarda el callback:   {temps_callback_ms:.1f} ms")
print(f"Prou ràpid? {'✅ Sí' if temps_callback_ms < temps_disponible_ms else '❌ No — glitch!'} ")
print()
print("Com ho solucionaria sense simplificar el càlcul?")
print("  Opció 1: augmentar blocksize (ex: 2048) → més temps disponible, però més latència")
print("  Opció 2: pre-calcular parts de l'eco fora del callback")
print("  Opció 3: usar un sample_rate més baix (ex: 22050)")

## 6. El codi real — per a Thonny

El codi de temps real d'aquesta sessió (pass-through, gain en temps real) està a:
- `exemples.py` — demo guiada
- `assignment.py` — el mini-repte

Executa'ls amb Thonny (entorn local). **No funcionaran a Colab.**

Un cop tinguis el codi funcionant a Thonny, puja el fitxer `assignment.py` completat al Classroom.